In [25]:
import os
import json
from datetime import datetime

# Define the base directory relative to the current script location
base_path = os.path.join(os.getcwd(), '..', 'chat_responses')

sql_query_list = []  # List to store all performance data entries

# Define the date range
start_date = datetime.strptime('2024-05-17', '%Y-%m-%d')
end_date = datetime.strptime('2024-06-02', '%Y-%m-%d')

# Iterate over each directory in the base path
for date_folder in os.listdir(base_path):
    # Convert folder name to date object
    try:
        folder_date = datetime.strptime(date_folder, '%Y-%m-%d')
    except ValueError:
        print(f"Skipping non-date folder: {date_folder}")
        continue
    
    # Check if the folder date is within the range
    if start_date <= folder_date <= end_date:
        date_folder_path = os.path.join(base_path, date_folder)
        
        # Check if the path is a directory
        if os.path.isdir(date_folder_path):
            # List all files in this directory
            for file_name in os.listdir(date_folder_path):
                print(f"\nFile name: {file_name}\n")
                
                if file_name.endswith('.json'):  # Ensure we are only reading JSON files
                    file_path = os.path.join(date_folder_path, file_name)

                    # Open and read the JSON file
                    try:
                        with open(file_path, 'r') as file:
                            data = json.load(file)
                            
                            if 'SQL Query' in data and 'SQL Data' in data:
                                question = data.get('User Question', '')
                                queries = data['SQL Query']

                                tables_list = ['vm_agg_season_player_opta', 'vm_agg_season_player_info', 'vt_season_stat_player']
                                
                                for num, query in enumerate(queries):
                                    for table in tables_list:
                                        if table in query:
                                            print(f"Query {num}: {query}")
                                            file_path = os.path.join(date_folder_path, file_name)
                                            print(f"File path: {file_path}")
                                            
                                            sql_data = data['SQL Data'][f'SQL Data {num}']
                                            
                                            print(f"Data: {sql_data}")
                                            print()
                                            info = {
                                                'question': question,
                                                'query': query,
                                                'data': sql_data,
                                            }
                                            sql_query_list.append(info)
                    except Exception as e:
                        print(f"Failed to read {file_name}: {str(e)}")

print(sql_query_list)



File name: response_0517_155040.json


File name: response_0517_135952.json


File name: response_0517_143950.json


File name: response_0517_155236.json


File name: response_0517_155830.json


File name: response_0517_124836.json

Query 1: SELECT player_id, non_penalty_goals FROM public.vt_season_stat_player WHERE season_id = 816527 ORDER BY non_penalty_goals DESC LIMIT 1
File path: /Users/svasquez/driblab/dl-vision/chat_bot/peformance/../chat_responses/2024-05-17/response_0517_124836.json
Data: [['player_id', 'non_penalty_goals', 'player_id', 'team_id', 'season_id', 'player_name', 'team_name'], [[290087, '19', 290087, 4272, 816527, 'Alexander Sorloth', 'Villarreal']]]


File name: response_0517_134820.json


File name: response_0517_111515.json

Query 0: SELECT saves_total FROM public.vt_season_stat_player WHERE season_id = 816529 AND player_id = (SELECT player_id FROM public.vt_season_info_player WHERE season_id = 816529 AND player_id = (SELECT player_id FROM t_player WHERE name =

In [26]:
sql_query_list

[{'question': "Who is the top scorer of La liga? Key information:[{'season': 'ESP I 2023', 'info': {'season_id': 816527, 'season_name': 'ESP I 2023'}}]",
  'query': 'SELECT player_id, non_penalty_goals FROM public.vt_season_stat_player WHERE season_id = 816527 ORDER BY non_penalty_goals DESC LIMIT 1',
  'data': [['player_id',
    'non_penalty_goals',
    'player_id',
    'team_id',
    'season_id',
    'player_name',
    'team_name'],
   [[290087, '19', 290087, 4272, 816527, 'Alexander Sorloth', 'Villarreal']]]},
 {'question': "how many saves has onana made in the premier league this season ? Key information:[{'season': 'ENG I 2023', 'info': {'season_id': 816529, 'season_name': 'ENG I 2023'}}]",
  'query': "SELECT saves_total FROM public.vt_season_stat_player WHERE season_id = 816529 AND player_id = (SELECT player_id FROM public.vt_season_info_player WHERE season_id = 816529 AND player_id = (SELECT player_id FROM t_player WHERE name = 'Onana')) AND saves_total IS NOT NULL",
  'data': [

In [27]:
len(sql_query_list)

72

In [28]:
with open('recent_sql_query_list.json', 'w') as f:
    json.dump(sql_query_list, f, indent=4)

In [31]:
print(os.getcwd())

/Users/svasquez/driblab/dl-vision/chat_bot/peformance


In [58]:
queries_path = 'recent_sql_query_list.json'

with open(queries_path, 'r') as f:
    queries = json.load(f)

test_queries = queries[0:10]

for query in test_queries:
    print(query['query'])
    print(query['data'])
    print()

SELECT player_id, non_penalty_goals FROM public.vt_season_stat_player WHERE season_id = 816527 ORDER BY non_penalty_goals DESC LIMIT 1
[['player_id', 'non_penalty_goals', 'player_id', 'team_id', 'season_id', 'player_name', 'team_name'], [[290087, '19', 290087, 4272, 816527, 'Alexander Sorloth', 'Villarreal']]]

SELECT saves_total FROM public.vt_season_stat_player WHERE season_id = 816529 AND player_id = (SELECT player_id FROM public.vt_season_info_player WHERE season_id = 816529 AND player_id = (SELECT player_id FROM t_player WHERE name = 'Onana')) AND saves_total IS NOT NULL
[['saves_total', 'player_id', 'team_id', 'season_id', 'player_name', 'team_name'], []]

SELECT player_id, goals FROM public.vt_season_stat_player WHERE season_id = 816527 ORDER BY goals DESC LIMIT 1
[['player_id', 'goals', 'player_id', 'team_id', 'season_id', 'player_name', 'team_name'], [[1598188, '20', 1598188, 20076, 816527, 'Artem Dovbyk', 'Girona FC']]]

SELECT player_id, SUM(dribbles_won) AS total_dribbles F

In [62]:
import os
import sys
from run_query import get_db_data, init_connection_pool, close_connection_pool
from add_keyinfo import add_key_info_new
from sql_val_run import validate_run_query

with open('/Users/svasquez/driblab/dl-vision/chat_bot/schemas/merged_table2col.json', 'r') as file:
    schema = json.load(file)

init_connection_pool()

data = []
for num , query in enumerate(test_queries):
    print(f"Query 0: {query['query']}")
    query_data = get_db_data(query['query'])
    query_data = add_key_info_new(query_data)

    test_queries[num]['data'] = query_data
    print(query_data)
    print()
close_connection_pool()

Database connection pool initialized at: 06-02 20:28:11
                                                                                
Query 0: SELECT player_id, non_penalty_goals FROM public.vt_season_stat_player WHERE season_id = 816527 ORDER BY non_penalty_goals DESC LIMIT 1
Fetching data from the database at: 06-02 20:28:11
Fetching data from the database at: 06-02 20:28:12
[['player_id', 'non_penalty_goals', 'player_name'], [[66, Decimal('23'), 'Best Forward']]]

Query 0: SELECT saves_total FROM public.vt_season_stat_player WHERE season_id = 816529 AND player_id = (SELECT player_id FROM public.vt_season_info_player WHERE season_id = 816529 AND player_id = (SELECT player_id FROM t_player WHERE name = 'Onana')) AND saves_total IS NOT NULL
Fetching data from the database at: 06-02 20:28:12
[['saves_total'], []]

Query 0: SELECT player_id, goals FROM public.vt_season_stat_player WHERE season_id = 816527 ORDER BY goals DESC LIMIT 1
Fetching data from the database at: 06-02 20:28:12
F

In [63]:
test_queries

[{'question': "Who is the top scorer of La liga? Key information:[{'season': 'ESP I 2023', 'info': {'season_id': 816527, 'season_name': 'ESP I 2023'}}]",
  'query': 'SELECT player_id, non_penalty_goals FROM public.vt_season_stat_player WHERE season_id = 816527 ORDER BY non_penalty_goals DESC LIMIT 1',
  'data': [['player_id', 'non_penalty_goals', 'player_name'],
   [[66, Decimal('23'), 'Best Forward']]]},
 {'question': "how many saves has onana made in the premier league this season ? Key information:[{'season': 'ENG I 2023', 'info': {'season_id': 816529, 'season_name': 'ENG I 2023'}}]",
  'query': "SELECT saves_total FROM public.vt_season_stat_player WHERE season_id = 816529 AND player_id = (SELECT player_id FROM public.vt_season_info_player WHERE season_id = 816529 AND player_id = (SELECT player_id FROM t_player WHERE name = 'Onana')) AND saves_total IS NOT NULL",
  'data': [['saves_total'], []]},
 {'question': 'Who is the top scorer of La liga?',
  'query': 'SELECT player_id, goals 